Logistic Regression
This notebook implements Logistic Regression for binary classification. It walks through a from-scratch implementation using gradient descent, compares it against scikit-learn, and evaluates the model on the breast-cancer dataset using accuracy and a confusion matrix.

Cell 1: Imports and Setup

In [ ]:
# Manasa Basavaraja
# Logistic Regression
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Reproducibility across runs
np.random.seed(42)

Cell 2: Dataset Loading

In [ ]:
# Use the built-in breast cancer dataset (binary: malignant vs benign).
# Sticking to a built-in dataset keeps this notebook self-contained.
data = load_breast_cancer()
X = data.data
y = data.target

print(f'Feature matrix shape: {X.shape}')
print(f'Target vector shape : {y.shape}')
print(f'Classes             : {data.target_names}')

# Quick peek at the first few rows as a dataframe.
dataframe = pd.DataFrame(X, columns=data.feature_names)
dataframe['target'] = y
dataframe.head()

Cell 3: Train / Test Split and Feature Scaling

In [ ]:
# Split into train and test sets. Stratify on y so both sets keep
# the same class balance.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Logistic Regression with gradient descent converges much faster when
# the features are standardized to zero mean and unit variance.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training samples: {X_train.shape[0]}')
print(f'Testing samples : {X_test.shape[0]}')

Cell 4: Logistic Regression from Scratch

In [ ]:
def sigmoid(z):
    """Numerically stable sigmoid function."""
    return 1.0 / (1.0 + np.exp(-z))


def fit_logistic_regression(X, y, learning_rate=0.1, epochs=1000):
    """
    Train a logistic regression classifier using batch gradient descent
    on the binary cross-entropy loss.

    Returns the weight vector, bias, and the loss recorded at each epoch.
    """
    m, n_features = X.shape

    # Initialize parameters to zero
    w = np.zeros(n_features)
    b = 0.0
    loss_history = []

    for epoch in range(epochs):
        # Forward pass: linear -> sigmoid
        z = X.dot(w) + b
        y_pred = sigmoid(z)

        # Binary cross-entropy loss with a small epsilon for numerical safety
        eps = 1e-15
        loss = -np.mean(y * np.log(y_pred + eps) + (1 - y) * np.log(1 - y_pred + eps))
        loss_history.append(loss)

        # Gradients
        error = y_pred - y
        dw = (1 / m) * X.T.dot(error)
        db = (1 / m) * np.sum(error)

        # Parameter update
        w -= learning_rate * dw
        b -= learning_rate * db

    return w, b, loss_history


w_scratch, b_scratch, loss_history = fit_logistic_regression(
    X_train_scaled, y_train, learning_rate=0.1, epochs=1000
)
print(f'Final training loss: {loss_history[-1]:.4f}')
print(f'Number of features : {len(w_scratch)}')

Cell 5: Loss Curve

In [ ]:
# Plot how the binary cross-entropy loss decreases as the model trains.
plt.plot(loss_history, color='#A52A2A')
plt.title('Gradient Descent: Cross-Entropy Loss vs. Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

Cell 6: Predictions from the Scratch Model

In [ ]:
def predict(X, w, b, threshold=0.5):
    """Convert probabilities to 0/1 class labels using a threshold."""
    probs = sigmoid(X.dot(w) + b)
    return (probs >= threshold).astype(int), probs


y_pred_scratch, y_probs_scratch = predict(X_test_scaled, w_scratch, b_scratch)

acc_scratch = accuracy_score(y_test, y_pred_scratch)
print(f'Scratch model accuracy: {acc_scratch:.4f}')

Cell 7: Logistic Regression using scikit-learn

In [ ]:
# Fit the same problem using sklearn so we can sanity-check the scratch model.
sk_model = LogisticRegression(max_iter=1000, random_state=42)
sk_model.fit(X_train_scaled, y_train)

y_pred_sklearn = sk_model.predict(X_test_scaled)
acc_sklearn = accuracy_score(y_test, y_pred_sklearn)
print(f'sklearn model accuracy: {acc_sklearn:.4f}')

Cell 8: Confusion Matrix

In [ ]:
# Confusion matrix for the scratch model.
cm = confusion_matrix(y_test, y_pred_scratch)
print('Confusion Matrix (scratch model):')
print(cm)

fig, ax = plt.subplots(figsize=(5, 4))
ax.matshow(cm, cmap='Blues')
for (i, j), value in np.ndenumerate(cm):
    ax.text(j, i, f'{value}', ha='center', va='center', color='black')
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(data.target_names)
ax.set_yticklabels(data.target_names)
plt.title('Confusion Matrix')
plt.show()

Cell 9: Classification Report

In [ ]:
# Precision, recall, and F1-score per class.
print('Scratch model:')
print(classification_report(y_test, y_pred_scratch, target_names=data.target_names))

print('sklearn model:')
print(classification_report(y_test, y_pred_sklearn, target_names=data.target_names))

# Note: Both implementations should land within ~1-2 percentage points of one
# another. Differences come from sklearn's solver (lbfgs by default) reaching
# a more exact optimum than vanilla batch gradient descent.